<a href="https://colab.research.google.com/github/Sciform/sciform-hwz-ai-in-controlling/blob/main/lecture_2/2_8_ts_global_dl_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Global Time Series Model with NHITS – Sales Forecasting for an E-Commerce Assortment

In the previous notebooks we trained **one model per time series** (a *local* model). In retail, however, we usually have to forecast
hundreds or thousands of products at once. A **global model** is trained on *all* time series of the assortment at the same time:

* the model shares its parameters across products,
* products with short or noisy histories profit from the patterns learned on the other products,
* only **one** model has to be trained, monitored and deployed.

**NHITS** (*Neural Hierarchical Interpolation for Time Series*, Challu et al. 2023) is a deep learning model for multi-step forecasting.
It is a pure feed-forward network (MLP) – no recurrence, no attention – and therefore fast to train, but it beats many much larger
models on standard benchmarks. We use the implementation in [neuralforecast](https://nixtlaverse.nixtla.io/neuralforecast/) (Nixtla).
neuralforecast models are **global by default**: the data frame just contains an additional column that identifies the time series,
and the training windows are sampled from all series.

**Agenda**

1. Load the *Online Retail* data set (UK online gift shop, Dec 2010 – Dec 2011)
2. Create sample data: daily revenue for an assortment of the 10 most regularly sold products
3. Baseline: seasonal naive forecast
4. Local vs. global NHITS model
5. Global NHITS with calendar features (weekday, UK holidays)
6. Evaluation per product and for the whole assortment (controlling view)
7. Final forecast for the next 4 weeks

## Setup

In [ ]:
# Install the required packages (neuralforecast is not pre-installed on Colab; PyTorch is)
%pip install -q neuralforecast holidays

In [ ]:
import logging
import warnings

import holidays
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

warnings.filterwarnings("ignore")
for name in ["pytorch_lightning", "lightning", "lightning.pytorch", "lightning_fabric", "neuralforecast"]:
    logging.getLogger(name).setLevel(logging.ERROR)   # keep the training output short

pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load the data

The data set contains all transactions of a UK-based online retailer selling unique all-occasion gifts. Many customers are wholesalers.

| column | meaning |
|---|---|
| InvoiceNo | invoice number, starts with `C` for cancellations |
| StockCode | product code |
| Description | product name |
| Quantity | quantity per transaction line |
| InvoiceDate | date and time of the invoice |
| UnitPrice | price per unit in GBP |
| CustomerID | customer number |
| Country | country of the customer |

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sciform/sciform-hwz-ai-in-controlling/main/data/online_retail.csv

In [ ]:
raw = pd.read_csv("online_retail.csv", dtype={"InvoiceNo": str, "StockCode": str}, encoding="ISO-8859-1")
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"], format="%m/%d/%Y %H:%M")
print(raw.shape)
raw.head()

### Data cleaning

* remove cancellations (`InvoiceNo` starting with `C`) and lines with non-positive quantity or price,
* remove non-product positions such as postage (`POST`), manual corrections (`M`), bank charges, Amazon fees, ... –
  real products have a stock code starting with five digits,
* compute the **revenue** per line: `Sales = Quantity * UnitPrice`.

In [ ]:
df = raw[~raw["InvoiceNo"].str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df[df["StockCode"].str.match(r"^\d{5}")]
df = df.assign(Sales=df["Quantity"] * df["UnitPrice"], Date=df["InvoiceDate"].dt.normalize())

print(f"{len(df):,} transaction lines, {df['StockCode'].nunique():,} products, "
      f"{df['Date'].min().date()} – {df['Date'].max().date()}")

## 2. Create sample data: the assortment

Most of the ~3,900 products are sold only sporadically. For this example we select the **10 products that are sold on the most days**
(regular sellers – "Schnelldreher"). This is our assortment. For each product we build a **daily revenue time series**:

* days without sales (e.g. Saturdays – the shop does not ship on Saturdays – and the Christmas break) get revenue 0,
* single bulk orders of wholesalers create extreme peaks. We cap each series at its 99 % quantile so that these one-off events do not
  dominate the training (in practice they would be planned separately, e.g. as key-account orders),
* the result is stored in *long format* with the columns `ds` (date), `ID` (product) and `y` (revenue) – exactly the format that
  neuralforecast expects for a global model (we tell it with `id_col="ID"` which column identifies the series).

In [ ]:
N_PRODUCTS = 10

selling_days = df.groupby("StockCode")["Date"].nunique().sort_values(ascending=False)
assortment = selling_days.head(N_PRODUCTS).index

# most frequent description per stock code
names = (df[df["StockCode"].isin(assortment)]
         .groupby("StockCode")["Description"].agg(lambda s: s.mode().iat[0]))

overview = pd.DataFrame({
    "Description": names,
    "SellingDays": selling_days[assortment],
    "Revenue_GBP": df[df["StockCode"].isin(assortment)].groupby("StockCode")["Sales"].sum(),
}).loc[assortment]
overview

In [ ]:
daily = (df[df["StockCode"].isin(assortment)]
         .groupby(["StockCode", "Date"])["Sales"].sum()
         .unstack("StockCode"))

# complete calendar: days without sales get revenue 0
full_range = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
daily = daily.reindex(full_range).fillna(0.0)

# cap one-off bulk orders at the 99% quantile of each product
daily = daily.clip(upper=daily.quantile(0.99), axis=1)

# long format: ds | ID | y
sales = (daily.rename_axis("ds").reset_index()
         .melt(id_vars="ds", var_name="ID", value_name="y")
         .sort_values(["ID", "ds"]).reset_index(drop=True))

sales.to_csv("online_retail_assortment_daily_sales.csv", index=False)   # sample data for later use
print(sales.shape)
sales.head()

In [ ]:
def plot_grid(frames, title, ncols=2):
    # frames: dict ID -> callable(ax)
    ids = list(frames)
    nrows = int(np.ceil(len(ids) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.6 * nrows), sharex=True)
    for ax, i in zip(axes.flat, ids):
        frames[i](ax)
        ax.set_title(f"{i} – {names[i][:35]}", fontsize=10)
        ax.grid(alpha=0.3)
    for ax in axes.flat[len(ids):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()

weekly = daily.resample("W").sum()
plot_grid({i: (lambda ax, i=i: ax.plot(weekly.index, weekly[i], color="tab:blue")) for i in assortment},
          "Weekly revenue per product (GBP) – sample data")

In [ ]:
# day-of-week pattern of the whole assortment (Mon=0 ... Sun=6) – note: no sales on Saturdays
daily.sum(axis=1).groupby(daily.index.dayofweek).mean().plot.bar(
    title="Average daily revenue of the assortment by weekday", xlabel="weekday (0 = Monday)", ylabel="GBP");

### Train / test split

We forecast the last **4 weeks (28 days)** of each product. All earlier data is used for training.
Because all series share the same calendar, a simple date cut-off gives the same split for every product.

> We have only ~1 year of data. NHITS looks at a window of the most recent days, so it cannot learn a yearly pattern such as
> the pre-Christmas peak in Nov/Dec – it can only react to the level it sees in its input window.

In [ ]:
HORIZON = 28
cutoff = sales["ds"].max() - pd.Timedelta(days=HORIZON)

df_train = sales[sales["ds"] <= cutoff].reset_index(drop=True)
df_test = sales[sales["ds"] > cutoff].reset_index(drop=True)
print(f"train: {df_train['ds'].min().date()} – {df_train['ds'].max().date()}  ({df_train['ds'].nunique()} days)")
print(f"test:  {df_test['ds'].min().date()} – {df_test['ds'].max().date()}  ({df_test['ds'].nunique()} days)")

### Error metrics

* **MAE** – mean absolute error in GBP per day,
* **RMSE** – root mean squared error, punishes large errors,
* **WAPE** – weighted absolute percentage error = Σ|y − ŷ| / Σ|y|. Unlike MAPE it also works with days of zero revenue
  and can be compared across products of different size – a common KPI in retail demand planning.

In [ ]:
def metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    err = y_true - y_pred
    return pd.Series({"MAE": np.mean(np.abs(err)),
                      "RMSE": np.sqrt(np.mean(err ** 2)),
                      "WAPE_%": 100 * np.sum(np.abs(err)) / np.sum(np.abs(y_true))})

def evaluate(pred, col="yhat", name="model"):
    # pred: data frame with ds, ID, y, <col>. Returns metrics per product + over all products.
    per_id = pred.groupby("ID").apply(lambda g: metrics(g["y"], g[col]))
    per_id.loc["ALL"] = metrics(pred["y"], pred[col])
    per_id.columns = pd.MultiIndex.from_product([[name], per_id.columns])
    return per_id

results = {}

## 3. Baseline: seasonal naive forecast

Every model has to beat a simple benchmark. The *seasonal naive* forecast repeats the revenue of the last observed week
for all four weeks of the horizon (same weekday → same value).

In [ ]:
last_week = df_train[df_train["ds"] > cutoff - pd.Timedelta(days=7)].assign(dow=lambda d: d["ds"].dt.dayofweek)
pred_naive = (df_test.assign(dow=lambda d: d["ds"].dt.dayofweek)
              .merge(last_week[["ID", "dow", "y"]].rename(columns={"y": "yhat"}), on=["ID", "dow"]))

results["seasonal_naive"] = evaluate(pred_naive, name="seasonal_naive")
results["seasonal_naive"].loc[["ALL"]]

## 4. Local vs. global NHITS model

### How NHITS works

NHITS maps a window of the last `input_size` days directly to the next `h` days (*multi-step forecast*, all 28 days at once):

```
last 56 days ──► Stack 1: MaxPool(4) ─► MLP ─►  4 points ─► interpolate to 28 days   (slow pattern: level / trend)
                 Stack 2: MaxPool(2) ─► MLP ─► 14 points ─► interpolate to 28 days   (medium frequencies)
                 Stack 3: no pooling ─► MLP ─► 28 points                              (fast pattern: weekday)
                 forecast = sum of the three stack outputs  (each stack works on the residual of the previous one)
```

* **Multi-rate input pooling** (`n_pool_kernel_size`): the first stacks see a smoothed, down-sampled version of the input.
* **Hierarchical interpolation** (`n_freq_downsample`): the first stacks predict only a few points that are interpolated over the
  horizon – they *must* model the slow components; the last stack adds the fine, high-frequency details.
  This makes the model efficient for long horizons and gives a rough decomposition into low and high frequencies.
* **Scaling** (`scaler_type="robust"`): every input window is normalised with its own median and spread. This is important for a global
  model, because the products have very different revenue levels – the network learns the common *shape* on the normalised scale.
* **Probabilistic forecast** (`loss=MQLoss(level=[80])`): the network is trained with the quantile loss and returns the median plus an
  80 % prediction interval (10 % and 90 % quantile).

### Local vs. global

| | training data | number of networks |
|---|---|---|
| local model | one product | **one NHITS per product** (10 trainings) |
| global model | all products (column `ID`) | **one** NHITS shared by all products |

> Training a neural network is random (initial weights, sampled windows). We fix `random_seed` to obtain reproducible results;
> with other seeds the numbers below change by a few percent.

In [ ]:
INPUT_SIZE = 2 * HORIZON      # the network looks at the last 8 weeks
MAX_STEPS = 500               # number of gradient steps

def build_nhits(alias, futr_exog_list=None, max_steps=MAX_STEPS):
    return NHITS(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_pool_kernel_size=[4, 2, 1],     # input pooling per stack
        n_freq_downsample=[7, 2, 1],      # stack 1: 4 points, stack 2: 14 points, stack 3: 28 points
        mlp_units=3 * [[256, 256]],
        loss=MQLoss(level=[80]),          # median + 80% prediction interval
        scaler_type="robust",             # normalise every window per series
        futr_exog_list=futr_exog_list,    # known future inputs (section 5)
        learning_rate=1e-3,
        max_steps=max_steps,
        batch_size=32,                    # number of series per batch
        windows_batch_size=256,           # number of training windows per batch
        random_seed=42,
        alias=alias,
        # quiet training without progress bars / logs
        enable_progress_bar=False, enable_model_summary=False, logger=False,
    )

def to_pred(fc, alias):
    # neuralforecast returns the columns <alias>-median, <alias>-lo-80, <alias>-hi-80
    fc = fc.rename(columns={f"{alias}-median": "yhat", f"{alias}-lo-80": "lo", f"{alias}-hi-80": "hi"})
    fc[["yhat", "lo", "hi"]] = fc[["yhat", "lo", "hi"]].clip(lower=0)   # no negative revenue
    return df_test.merge(fc[["ID", "ds", "yhat", "lo", "hi"]], on=["ID", "ds"])

In [ ]:
%%time
# local model: one network per product
fc_local = []
for i in assortment:
    nf = NeuralForecast(models=[build_nhits("nhits_local")], freq="D")
    nf.fit(df_train[df_train["ID"] == i], id_col="ID")
    fc_local.append(nf.predict())
pred_local = to_pred(pd.concat(fc_local), "nhits_local")

results["nhits_local"] = evaluate(pred_local, name="nhits_local")
results["nhits_local"].loc[["ALL"]]

In [ ]:
%%time
# global model: one network for all products
nf_global = NeuralForecast(models=[build_nhits("nhits_global")], freq="D")
nf_global.fit(df_train, id_col="ID")
pred_global = to_pred(nf_global.predict(), "nhits_global")

results["nhits_global"] = evaluate(pred_global, name="nhits_global")
results["nhits_global"].loc[["ALL"]]

## 5. Global NHITS with calendar features

So far the network only sees the past revenue. It has to *infer* from the input window which weekday comes next.
Calendar information is known in advance for every future day, so we can pass it as **future exogenous variables** (`futr_exog_list`):

* the weekday as one-hot encoding (`dow_0` … `dow_6`),
* a flag for UK public holidays (England) – including Christmas and New Year.

For the forecast, neuralforecast needs these columns for the 28 future days as well (`futr_df`).

In [ ]:
uk_holidays = holidays.UK(subdiv="England", years=range(2010, 2013))
DOW_COLS = [f"dow_{d}" for d in range(7)]
EXOG = DOW_COLS + ["holiday"]

def add_calendar(d):
    d = d.copy()
    for k in range(7):
        d[f"dow_{k}"] = (d["ds"].dt.dayofweek == k).astype(float)
    d["holiday"] = d["ds"].dt.date.map(lambda x: float(x in uk_holidays))
    return d

print(sorted(h for h in uk_holidays if pd.Timestamp("2010-12-01") <= pd.Timestamp(h) <= pd.Timestamp("2012-01-10")))

In [ ]:
%%time
nf_exog = NeuralForecast(models=[build_nhits("nhits_global_cal", futr_exog_list=EXOG)], freq="D")
nf_exog.fit(add_calendar(df_train), id_col="ID")

# future calendar for the 28 days of the horizon
futr = add_calendar(nf_exog.make_future_dataframe())
pred_exog = to_pred(nf_exog.predict(futr_df=futr), "nhits_global_cal")

results["nhits_global_cal"] = evaluate(pred_exog, name="nhits_global_cal")
results["nhits_global_cal"].loc[["ALL"]]

## 6. Evaluation

### Overall comparison (all products, 28-day horizon)

In [ ]:
comparison = pd.concat(results.values(), axis=1)
overall = comparison.loc["ALL"].unstack(0).T
overall

In [ ]:
overall["WAPE_%"].plot.barh(title="WAPE over the whole assortment (lower is better)", xlabel="WAPE in %");

In [ ]:
# store the test forecasts – notebook 2_9 compares them with a GenAI foundation model (Chronos)
test_predictions = (pred_global[["ds", "ID", "y", "yhat"]].rename(columns={"yhat": "nhits_global"})
                    .merge(pred_local[["ds", "ID", "yhat"]].rename(columns={"yhat": "nhits_local"}), on=["ds", "ID"])
                    .merge(pred_exog[["ds", "ID", "yhat"]].rename(columns={"yhat": "nhits_global_cal"}), on=["ds", "ID"]))
test_predictions.to_csv("nhits_test_predictions.csv", index=False)
test_predictions.head()

### Accuracy per product (WAPE in %)

In [ ]:
wape = comparison.xs("WAPE_%", axis=1, level=1)
wape.insert(0, "Description", names.reindex(wape.index).fillna("all products"))
wape.style.format(precision=1, subset=wape.columns[1:]).highlight_min(axis=1, subset=wape.columns[1:], color="lightgreen")

### Forecast vs. actuals per product (global model with 80 % prediction interval)

In [ ]:
hist = df_train[df_train["ds"] > cutoff - pd.Timedelta(days=42)]

def plot_product(ax, i):
    h, p, a = hist[hist["ID"] == i], pred_global[pred_global["ID"] == i], pred_exog[pred_exog["ID"] == i]
    ax.plot(h["ds"], h["y"], color="grey", lw=1, label="history")
    ax.plot(p["ds"], p["y"], color="black", lw=1.2, label="actual")
    ax.plot(p["ds"], p["yhat"], color="tab:blue", label="global")
    ax.fill_between(p["ds"], p["lo"], p["hi"], color="tab:blue", alpha=0.2, label="80% interval")
    ax.plot(a["ds"], a["yhat"], color="tab:orange", ls="--", label="global + calendar")
    ax.axvline(cutoff, color="red", lw=0.8)
    if i == assortment[0]:
        ax.legend(fontsize=8, loc="upper left")

plot_grid({i: (lambda ax, i=i: plot_product(ax, i)) for i in assortment}, "Daily revenue: forecast vs. actual (GBP)")

### Controlling view: revenue of the whole assortment

For budgeting we are often less interested in the single product than in the total. Product forecasts are summed up
(*bottom-up* forecast). Errors of single products partly cancel out, therefore the error of the total is much smaller.

In [ ]:
total = (pred_global[["ds", "y", "yhat"]].groupby("ds").sum()
         .join(pred_exog.groupby("ds")["yhat"].sum().rename("yhat_cal"))
         .join(pred_naive.groupby("ds")["yhat"].sum().rename("yhat_naive")))

ax = total[["y", "yhat", "yhat_cal", "yhat_naive"]].plot(figsize=(14, 4), style=["k-", "b-", "C1--", "g:"],
                                                        title="Daily revenue of the assortment (GBP)")
ax.legend(["actual", "global", "global + calendar", "seasonal naive"]); ax.grid(alpha=0.3)
plt.show()

summary = pd.DataFrame({
    "actual_GBP": [total["y"].sum()] * 3,
    "forecast_GBP": [total["yhat"].sum(), total["yhat_cal"].sum(), total["yhat_naive"].sum()],
}, index=["nhits_global", "nhits_global_cal", "seasonal_naive"])
summary["deviation_%"] = 100 * (summary["forecast_GBP"] / summary["actual_GBP"] - 1)
summary["daily_WAPE_%"] = [metrics(total["y"], total[c])["WAPE_%"] for c in ["yhat", "yhat_cal", "yhat_naive"]]
summary

### Interpretation

* All NHITS models beat the seasonal naive baseline (WAPE ≈ 85 %). The **global** model (WAPE ≈ 60 %) is clearly better than ten
  **local** models (≈ 75 %): a single product offers only ~260 training windows of 56 + 28 days, which is far too little for a neural
  network – the global model sees 10× more data and generalises much better. It is also trained ~10× faster.
  This is the main argument for global models in deep learning.
* The **calendar features** improve the accuracy only a little (≈ 58 %): with an input window of exactly 8 weeks the network can already
  read the weekday pattern (e.g. no revenue on Saturdays) from the past values. Known future inputs become more important for effects that
  are *not* visible in the recent history – holidays, promotions, price changes.
* Look at the **total revenue (deviation_%)**: the test period is the pre-Christmas peak (Nov/Dec). NHITS only sees the last 8 weeks and
  has never seen a Christmas season in a training window, so it **underestimates the total** by roughly a third, while the seasonal
  naive forecast already "sees" the high level of the last week. A typical trade-off a controller must be aware of:
  a lower daily error does not automatically mean a better *volume* forecast.
* The point forecast is the **median**. For skewed data such as daily revenue (many small days, a few very large ones) the median
  is below the mean – another reason why the summed forecast is too low. For budgeting, a forecast of the mean
  (e.g. `loss=MAE()` → median, `loss=MSE()` → mean) or a bias correction on the aggregated level can be the better choice.
* Training a neural network is random: with a different `random_seed` the WAPE changes by a few percentage points.
  Before choosing a model, always compare several seeds or use a backtest over several forecast origins (`nf.cross_validation`).

## 7. Final forecast for the next 4 weeks

For production use the selected model is retrained on **all** available data and forecasts the next 28 days for every product of the assortment.
The result can be exported e.g. to a planning or BI tool.

> Note: the data ends on 9 Dec 2011, so the forecast covers the Christmas break, in which the shop hardly sells anything. Through the
> holiday flag the model knows Christmas Day, Boxing Day and New Year, but it has never seen a complete Christmas break in its training
> windows – a planner would correct this period manually.

In [ ]:
%%time
nf_final = NeuralForecast(models=[build_nhits("nhits", futr_exog_list=EXOG)], freq="D")
nf_final.fit(add_calendar(sales), id_col="ID")

futr = add_calendar(nf_final.make_future_dataframe())
forecast = nf_final.predict(futr_df=futr)[["ds", "ID", "nhits-median", "nhits-lo-80", "nhits-hi-80"]]
forecast.columns = ["ds", "ID", "forecast", "lower_80", "upper_80"]
forecast[["forecast", "lower_80", "upper_80"]] = forecast[["forecast", "lower_80", "upper_80"]].clip(lower=0)  # no negative revenue
forecast["Description"] = forecast["ID"].map(names)

forecast.to_csv("assortment_sales_forecast.csv", index=False)
forecast.head(10)

In [ ]:
# planned revenue per product and week
plan = forecast.assign(week=forecast["ds"].dt.to_period("W")).pivot_table(
    index=["ID", "Description"], columns="week", values="forecast", aggfunc="sum")
plan["total"] = plan.sum(axis=1)
plan.loc[("ALL", "assortment"), :] = plan.sum()
plan.round(0)

## Summary and exercises

* A **global model** is trained once on all time series of the assortment. neuralforecast models are global by default – the data
  only needs an ID column (`id_col="ID"`).
* **NHITS** is a fast MLP-based model that forecasts the whole horizon at once. Input pooling and hierarchical interpolation split the
  forecast into slow (level/trend) and fast (weekday) components.
* Window-wise scaling (`scaler_type`) lets products of very different size share one network.
* Known future information such as weekday and holidays is passed as `futr_exog_list`; the quantile loss (`MQLoss`) provides
  prediction intervals.
* For controlling, the bottom-up aggregation of product forecasts has a much lower daily error than the single products –
  but check the bias of the total: effects that the model cannot see in its input window (here: the Christmas season) accumulate
  over the assortment.

**Exercises**

1. Increase the assortment to `N_PRODUCTS = 50`. How do the local and the global model compare now? How long does training take?
2. Forecast `Quantity` (units) instead of revenue – which is the better target for replenishment, which for budgeting?
3. Change `INPUT_SIZE` (e.g. 14, 28, 112 days) and `MAX_STEPS`. How sensitive is the accuracy? Repeat with different `random_seed`s.
4. Replace `NHITS` by `NBEATS` or `TFT` from `neuralforecast.models` (same interface) and compare the results.
5. Implement the business rule "no revenue on Saturdays" and measure the effect on WAPE.
6. Add the number of distinct customers per day of the whole shop as a historic exogenous variable (`hist_exog_list`).